# In-Vehicle People Sensing: Lightweight 1D-CNN Pipeline
This notebook implements a **lightweight 1D Convolutional Neural Network** for **people counting (multi-class)** using IR-UWB radar data.

### Key Features
- **No manual feature extraction** — learns directly from raw radar signals
- **File-level splitting** — no data leakage between train/test
- **Sanity checks** — label-shuffle test, confidence analysis, k-fold CV
- **Apple Metal GPU** support for accelerated training
- **Edge-deployable** — TFLite export for embedded systems

In [1]:
import os, gc, warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report)

# ── GPU ──
gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow {tf.__version__}")
if gpus:
    print(f"✅ GPU: {gpus}")
    print('⚠️ Disabling Apple Metal GPU due to tensorflow-metal down_cast abort bug!')
    try:
        tf.config.set_visible_devices([], 'GPU')
    except: pass
    gpus = []
else:
    print("⚠️  No GPU — CPU only")

# ── Platform ──
ON_KAGGLE = os.path.exists("/kaggle/input")
print(f"Platform: {'Kaggle' if ON_KAGGLE else 'Local'}")

def find_dataset_path():
    """
    Auto-detect radar dataset root regardless of what it is named.
    Strategy: walk /kaggle/input (or ./ locally), find the deepest
    directory that contains tab-separated files with >=1000 rows.
    Returns the highest-level directory that has such files.
    """
    search_roots = ["/kaggle/input"] if ON_KAGGLE else ["."]

    def looks_like_radar(filepath):
        """Quick check: tab-separated and has enough rows."""
        try:
            n, c = 0, 0
            with open(filepath) as f:
                for i, line in enumerate(f):
                    if i == 0:
                        parts = line.strip().split("\t")
                        if len(parts) < 10:   # radar has 400+ cols
                            return False
                        c = len(parts)
                    n += 1
                    if n > 1001: break      # enough to confirm row count
            return n >= 1000 and c >= 10
        except Exception:
            return False

    for search_root in search_roots:
        for root, dirs, files in os.walk(search_root):
            # Skip hidden dirs
            dirs[:] = [d for d in dirs if not d.startswith(".")]
            for fname in files:
                if fname.startswith(".") or fname.endswith((".png",".py",".md",".txt",".csv",".json",".yaml",".zip")):
                    continue
                fp = os.path.join(root, fname)
                if looks_like_radar(fp):
                    # Return parent directory of this file
                    # Walk up to find the dataset root
                    # (highest dir that still contains radar files)
                    return root
    return None

DATASET_PATH = find_dataset_path()

if DATASET_PATH is None:
    if ON_KAGGLE:
        print("❌ No radar dataset found! Datasets attached to this notebook:")
        for d in os.listdir("/kaggle/input"):
            print(f"   /kaggle/input/{d}")
            for sub in os.listdir(f"/kaggle/input/{d}")[:5]:
                print(f"      {sub}")
        raise FileNotFoundError(
            "Attach the In-Vehicle Radar dataset via Add Data in Kaggle."
        )
    else:
        raise FileNotFoundError(
            "Dataset not found locally. Set DATASET_PATH manually."
        )

# Walk up one level if needed (radar files may be in a subfolder)
# Find the highest ancestor that still contains radar data
parent = os.path.dirname(DATASET_PATH)
if ON_KAGGLE and parent.startswith("/kaggle/input") and parent != "/kaggle/input":
    # Use /kaggle/input/<dataset-name> as root so we get all subfolders
    parts = DATASET_PATH.replace("/kaggle/input/", "").split("/")
    DATASET_PATH = f"/kaggle/input/{parts[0]}"

print(f"📂 Dataset root: {DATASET_PATH}")

# ── Config ──
WINDOW_SIZE   = 1000
BATCH_SIZE    = 32
EPOCHS        = 30
LR            = 1e-3
N_CLASSES     = 6

# On Apple Silicon (Metal), GPU & CPU share RAM → use generator.
# On Kaggle (discrete GPU), tensor slices are fine and faster.
USE_GENERATOR = False
print(f"Training strategy: {'generator (Metal)' if USE_GENERATOR else 'tensor slices'}")

try:
    import psutil
    def mem(tag=""):
        gb = psutil.Process(os.getpid()).memory_info().rss / 1024**3
        print(f"  ▶ RAM {tag}: {gb:.2f} GB")
except ImportError:
    def mem(tag=""): pass

print("Ready."); mem("baseline")


TensorFlow 2.21.0
⚠️  No GPU — CPU only
Platform: Local


FileNotFoundError: Dataset not found locally. Set DATASET_PATH manually.

## Step 1: Load Dataset with File Tracking

In [ ]:
def get_label(root):
    r = root.lower()
    if 'empty' in r or 'baseline' in r or '0 people' in r: return 0
    if ('1 people' in r or 'baby lies' in r or 'child sits' in r
            or 'child lies' in r
            or ('adult sits in the rear seat' in r and 'driver' not in r)
            or ('adult lies in the rear seat' in r and 'driver' not in r)): return 1
    if '2 people' in r or 'driver seat' in r: return 2
    if '3 people' in r: return 3
    if '4 people' in r: return 4
    if '5 people' in r: return 5
    return 1

def load_radar_dataset(base_path, window_size=1000):
    """Memory-efficient two-pass loader (float32, pre-allocated array)."""

    # ── Pass 1: scan files ──
    print(f'Scanning {base_path} ...')
    valid = []
    for root, _, files in os.walk(base_path):
        for fname in sorted(files):
            if fname.startswith('.') or fname.endswith(('.png','.py','.md')): continue
            fp = os.path.join(root, fname)
            try:
                n_lines, n_cols = 0, 0
                with open(fp) as f:
                    for i, line in enumerate(f):
                        if i == 0: n_cols = len(line.strip().split('\t'))
                        n_lines += 1
                if n_lines >= window_size and n_cols >= 2:
                    valid.append((fp, get_label(root), n_lines, n_cols))
            except Exception:
                pass

    if not valid:
        print(f'\n❌ No valid radar files found in: {base_path}')
        print('   • Files must be tab-separated with ≥2 columns and ≥1000 rows')
        print(f'   • Directory contents:')
        for root, dirs, files in os.walk(base_path):
            level = root.replace(base_path, '').count(os.sep)
            if level < 2:
                indent = '     ' + '  ' * level
                print(f'{indent}{os.path.basename(root)}/')
                if level == 1:
                    for fn in files[:3]:
                        print(f'{indent}  {fn}')
        raise ValueError(f'No valid files found under {base_path}')

    print(f'  Found {len(valid)} valid files')

    # ── Pass 2: pre-allocate + fill ──
    n_sig    = valid[0][3] - 1           # columns minus timestamp
    wins_per = [(n - window_size) // window_size + 1 for _, _, n, _ in valid]
    total    = sum(wins_per)
    size_gb  = total * window_size * n_sig * 4 / 1024**3
    print(f'  {total} windows | shape ({window_size},{n_sig}) | {size_gb:.2f} GB float32')

    X      = np.empty((total, window_size, n_sig), dtype=np.float32)
    y      = np.empty(total, dtype=np.int64)
    groups = np.empty(total, dtype=np.int64)
    names  = []
    idx    = 0

    for fid, (fp, lbl, n_rows, _) in enumerate(valid):
        try:
            data = np.loadtxt(fp, delimiter='\t', dtype=np.float32)
        except Exception as e:
            print(f'  ⚠️  Skip {os.path.basename(fp)}: {e}')
            continue
        sig = data[:, 1:]
        for w in range(wins_per[fid]):
            s = w * window_size
            X[idx] = sig[s:s + window_size]
            y[idx] = lbl
            groups[idx] = fid
            idx += 1
        names.append(os.path.relpath(fp, base_path))
        del data, sig

    return X[:idx], y[:idx], groups[:idx], names

X, y, groups, fnames = load_radar_dataset(DATASET_PATH, WINDOW_SIZE)
n_files = len(np.unique(groups))
print(f'\nLoaded {len(X)} windows from {n_files} files')
print(f'Window shape : {X[0].shape}')
print(f'Class dist   : {dict(zip(*np.unique(y, return_counts=True)))}')
mem('after load')

print(f"\n{'File':>4} | {'Win':>5} | {'Cls':>3} | Path")
print('-'*65)
for fid in np.unique(groups):
    mask = groups == fid
    print(f'{fid:>4} | {np.sum(mask):>5} | {y[mask][0]:>3} | {fnames[fid]}')


Scanning ./In-Vehicle-Radar-Data/People Counting in the Moving Vehicle/2 people ...
  Found 10 valid files
  60 windows | shape (1000,437) | 0.10 GB float32

Loaded 60 windows from 10 files
Window shape : (1000, 437)
Class dist   : {2: 60}
  ▶ RAM after load: 0.49 GB

File |   Win | Cls | Path
-----------------------------------------------------------------
   0 |     6 |   2 | 2020-05-24-19-11-15
   1 |     6 |   2 | 2020-05-24-19-13-05
   2 |     6 |   2 | 2020-05-24-19-14-53
   3 |     6 |   2 | 2020-05-24-19-16-41
   4 |     6 |   2 | 2020-05-24-19-18-28
   5 |     6 |   2 | 2020-05-24-19-20-41
   6 |     6 |   2 | 2020-05-24-19-22-29
   7 |     6 |   2 | 2020-05-24-19-24-19
   8 |     6 |   2 | 2020-05-24-19-26-08
   9 |     6 |   2 | 2020-05-24-19-28-27


## Step 2: File-Level Split & Normalize

In [ ]:
unique_files = np.unique(groups)
file_labels  = np.array([y[groups==fid][0] for fid in unique_files])
print(f'Files: {len(unique_files)} | per class: {dict(zip(*np.unique(file_labels, return_counts=True)))}')

tv_files, test_files = train_test_split(
    unique_files, test_size=0.2, random_state=42, stratify=file_labels)
tv_labels = np.array([y[groups==fid][0] for fid in tv_files])
train_files, val_files = train_test_split(
    tv_files, test_size=0.19, random_state=42, stratify=tv_labels)

train_mask = np.isin(groups, train_files)
val_mask   = np.isin(groups, val_files)
test_mask  = np.isin(groups, test_files)

X_train, y_train = X[train_mask].copy(), y[train_mask]
X_val,   y_val   = X[val_mask].copy(),   y[val_mask]
X_test,  y_test  = X[test_mask].copy(),  y[test_mask]

# Free original immediately
del X; gc.collect()
print('🧹 X freed'); mem('after free')

# Per-sample normalise in-place (no extra copy)
def norm_inplace(arr):
    mu  = arr.mean(axis=(1,2), keepdims=True)
    sig = arr.std(axis=(1,2),  keepdims=True) + 1e-8
    arr -= mu
    arr /= sig

for arr in [X_train, X_val, X_test]:
    norm_inplace(arr)

print(f'Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}')
print(f'Train classes : {dict(zip(*np.unique(y_train, return_counts=True)))}')
assert not (set(train_files) & set(test_files)), 'LEAK!'
assert not (set(train_files) & set(val_files)),  'LEAK!'
assert not (set(val_files)   & set(test_files)), 'LEAK!'
print('✅ No split overlap'); mem('after split')


Files: 10 | per class: {2: 10}
🧹 X freed
  ▶ RAM after free: 0.69 GB
Train: 36 | Val: 12 | Test: 12
Train classes : {2: 36}
✅ No split overlap
  ▶ RAM after split: 0.69 GB


## Step 3: Build & Train the 1D-CNN

In [ ]:
def build_1d_cnn(input_shape):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(16, 7, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(4),
        layers.Conv1D(32, 5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(4),
        layers.Conv1D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(4),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(0.5),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(N_CLASSES, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_1d_cnn(X_train.shape[1:])
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 1000, 16)       │        48,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1000, 16)       │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 250, 16)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 250, 32)        │         2,592 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 250, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 62, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 62, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 62, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 15, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,486 (236.27 KB)

 Trainable params: 60,262 (235.40 KB)

 Non-trainable params: 224 (896.00 B)

Model size: 236.3 KB


: 

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import Sequence
from tensorflow.keras.utils import to_categorical
import math

class DataSequence(Sequence):
    def __init__(self, x_set, y_set, batch_size, shuffle=False):
        self.x, self.y = x_set, y_set
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.x))
        if self.shuffle: np.random.shuffle(self.indices)
    def __len__(self):
        return math.ceil(len(self.x) / self.batch_size)
    def __getitem__(self, idx):
        inds = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        # Return cleanly as float32 inputs and float32 categorical targets!
        return self.x[inds].astype(np.float32), to_categorical(self.y[inds], num_classes=N_CLASSES).astype(np.float32)
    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

def make_dataset(X_arr, y_arr, shuffle=False, batch_size=BATCH_SIZE):
    return DataSequence(X_arr, y_arr, batch_size, shuffle)

train_ds = make_dataset(X_train, y_train, shuffle=True)
val_ds   = make_dataset(X_val,   y_val)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=7,
                                   restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                       patience=3, min_lr=1e-6, verbose=1),
]

print('▶ Training ...'); mem('before fit')

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
print('✅ Done!'); mem('after fit')


Class weights: {2: 1.0}
▶ Training ...
  ▶ RAM before fit: 0.73 GB
Epoch 1/30


## Step 4: Evaluate on Test Set

In [ ]:
test_ds     = make_dataset(X_test, y_test)
y_pred_prob = model.predict(test_ds, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
test_acc    = accuracy_score(y_test, y_pred)

print(f'Test Accuracy: {test_acc*100:.2f}%')
print(f'\n{classification_report(y_test, y_pred)}')


## Step 5: 🔍 Sanity Checks — Is 100% Accuracy Real?
Three tests to verify the model is truly learning meaningful patterns.

In [ ]:
print('='*60)
print('  SANITY CHECK 1: Prediction Confidence')
print('='*60)

max_probs = np.max(y_pred_prob, axis=1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(max_probs, bins=25, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(1/N_CLASSES, color='red', ls='--', label=f'Random (1/{N_CLASSES})')
axes[0].set_title('Max Softmax Confidence', fontweight='bold')
axes[0].set_xlabel('Max Predicted Probability'); axes[0].legend()

cmap = plt.cm.get_cmap('tab10', N_CLASSES)
for cls in np.unique(y_test):
    m = y_test == cls
    axes[1].hist(max_probs[m], bins=20, alpha=0.5, color=cmap(cls),
                 label=f'{cls} People', edgecolor='black')
axes[1].axvline(1/N_CLASSES, color='red', ls='--')
axes[1].set_title('Confidence by True Class', fontweight='bold')
axes[1].set_xlabel('Max Predicted Probability'); axes[1].legend()
plt.tight_layout(); plt.show()

cor = max_probs[y_pred == y_test]; inc = max_probs[y_pred != y_test]
print(f'Overall mean conf:    {max_probs.mean():.4f}')
if len(cor): print(f'Correct preds conf:   {cor.mean():.4f}')
if len(inc): print(f'Incorrect preds conf: {inc.mean():.4f}')
else: print('All predictions correct!')


In [ ]:
print('='*60)
print('  SANITY CHECK 2: Shuffled Labels Test')
print('='*60); mem('before shuffled test')

y_shuf = y_train.copy()
np.random.seed(42); np.random.shuffle(y_shuf)

m2 = build_1d_cnn(X_train.shape[1:])
m2.fit(make_dataset(X_train, y_shuf, shuffle=True), epochs=10, verbose=0)

y_shuf_pred  = np.argmax(m2.predict(make_dataset(X_test, y_test), verbose=0), axis=1)
shuffled_acc = accuracy_score(y_test, y_shuf_pred)
rnd_baseline = max(np.bincount(y_test)) / len(y_test)

del m2, y_shuf, y_shuf_pred
keras.backend.clear_session(); gc.collect(); mem('after shuffled test')

print(f'Real labels accuracy:     {test_acc*100:.2f}%')
print(f'Shuffled labels accuracy: {shuffled_acc*100:.2f}%')
print(f'Random baseline:          {rnd_baseline*100:.1f}%')
if shuffled_acc > rnd_baseline + 0.1:
    print('\n❌ WARNING: Shuffled model above chance — check for data leakage.')
else:
    print('\n✅ PASS: Shuffled model near chance — real signal learned.')


In [ ]:
from sklearn.model_selection import GroupKFold
print('='*60)
print('  SANITY CHECK 3: Grouped 3-Fold Cross-Validation')
print('='*60)

# Free train/val to make room for CV
del X_train, X_val; gc.collect(); mem('freed train/val')

# Reload raw data
X_cv, y_cv, g_cv, _ = load_radar_dataset(DATASET_PATH, WINDOW_SIZE)
mem('after CV reload')

gkf = GroupKFold(n_splits=3)
fold_acc, fold_details = [], []

for fi, (tr, te) in enumerate(gkf.split(X_cv, y_cv, g_cv)):
    print(f'  Fold {fi+1}/3: ', end='')
    Xtr, ytr = X_cv[tr].copy(), y_cv[tr]
    Xte, yte = X_cv[te].copy(), y_cv[te]
    norm_inplace(Xtr); norm_inplace(Xte)

    mc = build_1d_cnn(Xtr.shape[1:])
    mc.fit(make_dataset(Xtr, ytr, shuffle=True), epochs=10, verbose=0)
    preds = np.argmax(mc.predict(make_dataset(Xte, yte), verbose=0), axis=1)
    a = accuracy_score(yte, preds)
    fold_acc.append(a)
    nf = len(np.unique(g_cv[te]))
    fold_details.append((fi+1, nf, len(yte), a))
    print(f'files={nf}, windows={len(yte)}, acc={a*100:.1f}%')

    del mc, Xtr, Xte, ytr, yte, preds
    keras.backend.clear_session(); gc.collect(); mem(f'  fold {fi+1} done')

del X_cv, y_cv, g_cv; gc.collect()

# Restore X_train / X_val
X_all, y_all, g_all, _ = load_radar_dataset(DATASET_PATH, WINDOW_SIZE)
X_train = X_all[np.isin(g_all, train_files)].copy(); norm_inplace(X_train)
X_val   = X_all[np.isin(g_all, val_files)].copy();   norm_inplace(X_val)
del X_all, y_all, g_all; gc.collect()
train_ds = make_dataset(X_train, y_train, shuffle=True)
val_ds   = make_dataset(X_val, y_val)

mean_acc = np.mean(fold_acc); std_acc = np.std(fold_acc)
print(f"\n{'='*60}")
print(f'  3-Fold CV: {mean_acc*100:.2f}% ± {std_acc*100:.2f}%')
print(f"{'='*60}")
if mean_acc >= 0.99:   print('\n🎯 Genuinely easy task.')
elif mean_acc >= 0.90: print('\n✅ Strong performance.')
else:                  print('\n⚠️  Investigate further.')


## Step 6: Visualize All Results

In [ ]:
class_names   = ['0 People','1 Person','2 People','3 People','4 People','5 People']
present_cls   = np.unique(np.concatenate([y_test, y_pred]))
present_names = [class_names[c] for c in present_cls]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(history.history['accuracy'],     label='Train', lw=2)
axes[0,0].plot(history.history['val_accuracy'], label='Val',   lw=2)
axes[0,0].set_title('Accuracy', fontweight='bold')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(history.history['loss'],     label='Train', lw=2)
axes[0,1].plot(history.history['val_loss'], label='Val',   lw=2)
axes[0,1].set_title('Loss', fontweight='bold')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

cm   = confusion_matrix(y_test, y_pred, labels=present_cls)
disp = ConfusionMatrixDisplay(cm, display_labels=present_names)
disp.plot(ax=axes[1,0], cmap='Blues', colorbar=False)
axes[1,0].set_title('Confusion Matrix', fontweight='bold')

if fold_details:
    fnums = [d[0] for d in fold_details]; accs = [d[3]*100 for d in fold_details]
    axes[1,1].bar(range(len(accs)), accs, color='steelblue', edgecolor='black', alpha=0.8)
    axes[1,1].set_xticks(range(len(accs)))
    axes[1,1].set_xticklabels([f'Fold {n}' for n in fnums])
    axes[1,1].set_ylim(0, 110)
    axes[1,1].axhline(mean_acc*100, color='navy', ls='--', label=f'Mean {mean_acc*100:.1f}%')
    axes[1,1].set_title('3-Fold CV Accuracy', fontweight='bold'); axes[1,1].legend()

plt.suptitle(f'1D-CNN | Test {test_acc*100:.1f}% | CV {mean_acc*100:.1f}%±{std_acc*100:.1f}%',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## Step 7: Export for Deployment

In [ ]:
model.save('1d_cnn_human_detection.keras')
saved_size = os.path.getsize('1d_cnn_human_detection.keras') / 1024

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = converter.convert()
with open('1d_cnn_human_detection.tflite', 'wb') as f: f.write(tflite)
tflite_size = os.path.getsize('1d_cnn_human_detection.tflite') / 1024

print('='*55)
print('  DEPLOYMENT SUMMARY')
print('='*55)
print(f'  Keras model:        {saved_size:.1f} KB')
print(f'  TFLite (quantized): {tflite_size:.1f} KB')
print(f'  Parameters:         {model.count_params():,}')
print(f'  Test accuracy:      {test_acc*100:.2f}%')
print(f'  CV accuracy:        {mean_acc*100:.2f}% ± {std_acc*100:.2f}%')
print(f'  Platform:           {"Kaggle" if ON_KAGGLE else "Local"}')
print(f'  Trained on:         {"GPU" if gpus else "CPU"}')
print('='*55)


In [ ]:
# --- Visualize Training History (Accuracy and Loss) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange', linewidth=2)
axes[0].set_title('Model Accuracy over Epochs', fontweight='bold', fontsize=14)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].legend(loc='lower right')
axes[0].grid(True, linestyle='--', alpha=0.7)

axes[1].plot(history.history['loss'], label='Train Loss', color='blue', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', color='orange', linewidth=2)
axes[1].set_title('Model Loss over Epochs', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Sparse Categorical Crossentropy Loss', fontsize=12)
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
# --- Visualize Multi-Class Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
present_classes = np.unique(y_test)
labels = [f"{c} People" for c in present_classes]
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(cmap=plt.cm.Blues, ax=ax, values_format='d', colorbar=True)

plt.title('Confusion Matrix on Test Set', fontweight='bold', fontsize=14)
plt.xlabel('Predicted Count', fontsize=12)
plt.ylabel('True Count', fontsize=12)
plt.grid(False)
plt.show()


In [ ]:
import tf2onnx
import onnx
import tensorflow as tf
# Convert Keras model → ONNX
input_signature = [tf.TensorSpec(shape=(None, 1000, X_train.shape[2]), 
                                  dtype=tf.float32, name="input")]
onnx_model, _ = tf2onnx.convert.from_keras(model, 
                                             input_signature=input_signature,
                                             opset=17)
onnx.save(onnx_model, "1d_cnn_human_detection.onnx")
